# Build an agent with short-term memory on Databricks Lakebase

This notebook shows how to give an AI agent durable, thread-based **short-term memory** using **Databricks Lakebase** — a fully managed Postgres database — as the agent's checkpoint store.

Without memory, every call to an LLM endpoint is stateless: the agent has no idea what was said one message ago. By plugging Lakebase into LangGraph as a checkpointer, every turn of a conversation is automatically saved to Postgres under a **thread ID**. The next turn only needs to send that thread ID — not the full conversation history — and the agent picks up exactly where it left off.

**In this notebook you will:**
1. Create the Lakebase checkpoint tables (one-time setup)
2. Author a LangGraph agent that uses Lakebase as its checkpointer, wrapped in MLflow's `ResponsesAgent` interface for compatibility with Databricks Agent Framework
3. Test short-term memory locally — same thread → the agent remembers; new or missing thread → the agent forgets
4. Understand the next steps to register and deploy this agent so memory keeps working in production

## Prerequisites
- A running Lakebase instance. See **Get a Postgres database** ([AWS](https://docs.databricks.com/aws/en/oltp/projects/get-started) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/projects/get-started)). Lakebase is a fully managed, autoscaling Postgres OLTP database built into your Databricks workspace.
- Complete every `TODO` in this notebook with the name of your own Lakebase instance.

Source: [AI agent memory (Model Serving)](https://docs.databricks.com/aws/en/generative-ai/agent-framework/stateful-agents-model-serving)

### Install dependencies

In [0]:
%pip install -U -qqqq \
  "databricks-langchain[memory]==0.19.0" \
  "langgraph>=1.1.3" \
  "mlflow-skinny[databricks]>=3.10.1" \
  "psycopg[binary,pool]>=3.2.10" \
  "databricks-sdk>=0.61.0" \
  uv databricks-agents
dbutils.library.restartPython()

## Step 1 — One-time setup: create the Lakebase checkpoint tables

LangGraph's Postgres checkpointer needs a small set of tables (checkpoints + pending writes) to store conversation state. `CheckpointSaver` from `databricks_langchain` manages a connection pool to your Lakebase instance and creates those tables for you.

Run this cell **once per Lakebase instance** — not every time you deploy the agent.

In [0]:
from databricks_langchain import CheckpointSaver


# TODO: replace with the name of your own Lakebase instance
project_id = "demo-lake-memory"
branch_id = "development"
endpoint_id = "primary"
LAKEBASE_ENDPOINT_NAME = f'projects/{project_id}/branches/{branch_id}/endpoints/{endpoint_id}'

with CheckpointSaver(autoscaling_endpoint=LAKEBASE_ENDPOINT_NAME) as saver:
    saver.setup()
    print("Checkpoint tables ready.")

## Step 2 — Define the agent in code

### Write the agent to `agent.py`
The cell below uses `%%writefile` to save the agent definition to a local file, so it can be logged and deployed afterwards.

### What makes this agent stateful
Three things, working together:
- A **LangGraph `StateGraph`** that defines the agent's reasoning loop (call the model, optionally call tools, repeat).
- A **`CheckpointSaver(autoscaling_endpoint=lakebase_endpoint_name)`** passed in as the graph's `checkpointer`. Every time the graph runs a step, it durably snapshots `AgentState` to Lakebase, keyed by `thread_id`.
- A **`thread_id`**, carried in `custom_inputs`, that tells the checkpointer which conversation's history to load before answering, and where to save the new turn afterwards.

### Wrapping with `ResponsesAgent`
`LangGraphResponsesAgent` implements MLflow's [`ResponsesAgent`](https://www.mlflow.org/docs/latest/llms/responses-agent-intro/) interface, which Databricks recommends for multi-turn conversational agents. It resolves the `thread_id` for each request (from `custom_inputs`, from chat context, or a freshly generated UUID), then opens a short-lived Lakebase connection to run the graph for that turn.

In [0]:
%%writefile agent.py
import logging
import os
import uuid
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    CheckpointSaver,
)
from databricks.sdk import WorkspaceClient
from langchain_core.messages import AIMessage, AIMessageChunk, AnyMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
)

logger = logging.getLogger(__name__)
logging.basicConfig(level=os.getenv("LOG_LEVEL", "INFO"))

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to the ongoing conversation history, "
    "durably stored in Databricks Lakebase. Use prior turns in this thread as context, "
    "and use the available tools to answer questions."
)

############################################
# Lakebase configuration
############################################
# TODO: replace with the name of your own Lakebase instance.
# This must be the same instance you used to create the checkpoint tables in Step 1.
project_id = "demo-lake-memory"
branch_id = "development"
endpoint_id = "primary"
LAKEBASE_ENDPOINT_NAME = f'projects/{project_id}/branches/{branch_id}/endpoints/{endpoint_id}'

###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## beyond text generation.
## See https://docs.databricks.com/en/generative-ai/agent-framework/agent-tool.html
###############################################################################
tools = []

# Example UC tools; add your own as needed
UC_TOOL_NAMES: list[str] = []
if UC_TOOL_NAMES:
    uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
    tools.extend(uc_toolkit.tools)

#####################
## Define agent logic
#####################

class AgentState(TypedDict):
    """The state LangGraph checkpoints to Lakebase on every step, keyed by thread_id."""
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]


class LangGraphResponsesAgent(ResponsesAgent):
    """Stateful agent using ResponsesAgent with Lakebase-backed checkpointing for short-term memory."""

    def __init__(self, lakebase_endpoint_name: str):
        self.workspace_client = WorkspaceClient()
        self.lakebase_endpoint_name = lakebase_endpoint_name

        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
        self.system_prompt = SYSTEM_PROMPT
        self.model_with_tools = self.model.bind_tools(tools) if tools else self.model

    def _create_graph(self, checkpointer: Any):
        """Build the LangGraph StateGraph, attaching the Lakebase checkpointer."""

        def should_continue(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1]
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "continue"
            return "end"

        preprocessor = (
            RunnableLambda(lambda state: [{"role": "system", "content": self.system_prompt}] + state["messages"])
            if self.system_prompt
            else RunnableLambda(lambda state: state["messages"])
        )
        model_runnable = preprocessor | self.model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            return {"messages": [response]}

        workflow = StateGraph(AgentState)
        workflow.add_node("agent", RunnableLambda(call_model))

        if tools:
            workflow.add_node("tools", ToolNode(tools))
            workflow.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
            workflow.add_edge("tools", "agent")
        else:
            workflow.add_edge("agent", END)

        workflow.set_entry_point("agent")
        # Lakebase checkpointer: every step's state is saved to Postgres, keyed by thread_id
        return workflow.compile(checkpointer=checkpointer)

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        """Resolve the thread_id that selects which conversation's memory to use.

        Priority:
        1. thread_id passed explicitly in custom_inputs
        2. conversation_id from chat context (e.g. Playground, Review App)
        3. a brand-new UUID, which starts a fresh conversation with no memory
        """
        ci = dict(request.custom_inputs or {})

        if "thread_id" in ci:
            return ci["thread_id"]

        # https://mlflow.org/docs/latest/api_reference/python_api/mlflow.types.html#mlflow.types.agent.ChatContext
        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id

        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        thread_id = self._get_or_create_thread_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        # Convert incoming Responses messages to ChatCompletions format;
        # LangChain converts ChatCompletions -> LangChain message format automatically.
        langchain_msgs = self.prep_msgs_for_cc_llm([i.model_dump() for i in request.input])
        checkpoint_config = {"configurable": {"thread_id": thread_id}}

        # Borrow a pooled connection to Lakebase for the lifetime of this turn
        with CheckpointSaver(autoscaling_endpoint=self.lakebase_endpoint_name) as checkpointer:
            graph = self._create_graph(checkpointer)

            for event in graph.stream(
                {"messages": langchain_msgs},
                checkpoint_config,
                stream_mode=["updates", "messages"],
            ):
                if event[0] == "updates":
                    for node_data in event[1].values():
                        if len(node_data.get("messages", [])) > 0:
                            yield from output_to_responses_items_stream(node_data["messages"])
                elif event[0] == "messages":
                    try:
                        chunk = event[1][0]
                        if isinstance(chunk, AIMessageChunk) and chunk.content:
                            yield ResponsesAgentStreamEvent(
                                **self.create_text_delta(delta=chunk.content, item_id=chunk.id),
                            )
                    except Exception as exc:
                        logger.error("Error streaming chunk: %s", exc)


# ----- Export model -----
mlflow.langchain.autolog()
AGENT = LangGraphResponsesAgent(LAKEBASE_ENDPOINT_NAME)
mlflow.models.set_model(AGENT)

# Test the agent's short-term memory locally

Restart Python so the freshly written `agent.py` is picked up cleanly.

In [0]:
dbutils.library.restartPython()

### Turn 1 — start a new conversation
No `thread_id` is supplied, so the agent generates a brand-new one and there is nothing to remember yet.

In [0]:
from agent import AGENT
role = "user"
first_met_query = "Hi, my name is Oliver and I a working on agent memory."
response_1 = AGENT.predict({
    "input": [{"role": role, "content": first_met_query}]
})
print(response_1.model_dump(exclude_none=True))
thread_id = response_1.custom_outputs["thread_id"]

### Turn 2 — same thread_id → short-term memory kicks in
By passing the same `thread_id` back in, the agent loads the checkpointed history from Lakebase before answering. It correctly remembers what was said in turn 1.

In [0]:
def format_short_response(response,size=120) :
    return response.model_dump(exclude_none=True)["output"][0]["content"][0]["text"][:size]

In [0]:
query = "Do you know anything about me ?"
response_2 = AGENT.predict({
    "input": [{"role": role, "content": query}],
    "custom_inputs": {"thread_id": thread_id}
})
print("Response 2:", format_short_response(response_2))

### Turn 3 — no thread_id → no memory
Omitting the thread_id starts a brand-new, empty thread. The agent has no checkpointed history to load, so it can't answer the same question.

In [0]:
response_3 = AGENT.predict({
    "input": [{"role": "user", "content": query}],
})
print("Response 3, no thread_id passed:", format_short_response(response_3))

### Using ChatContext's conversation_id as the thread_id
Clients that automatically pass a [`ChatContext`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.types.html#mlflow.types.agent.ChatContext) — such as the Playground or the Review App — don't need to manage `thread_id` manually. The agent falls back to `conversation_id` from the chat context, so memory keeps working transparently.

In [0]:
from agent import AGENT
from mlflow.types.responses import ResponsesAgentRequest, ChatContext

conversation_id = "e396d36f-b237-484f-ad6e-f000551703f5"

req = ResponsesAgentRequest(
    input=[{"role": role, "content": first_met_query}],
    context=ChatContext(
        conversation_id=conversation_id,
        user_id="oliver@databricks.com"
    )
)
result = AGENT.predict(req)

print(result.model_dump(exclude_none=True))
thread_id = result.custom_outputs["thread_id"]
print(f"Resolved thread_id from agent: {thread_id}")

## Key takeaways
- **Lakebase is the durable store**, not the agent logic. `CheckpointSaver(instance_name=...)` is the only piece of Lakebase-specific code the agent needs.
- **`thread_id` is the unit of memory.** Same `thread_id` → conversation continues. New or missing `thread_id` → a clean slate.
- **Clients don't need to send full history.** Only the latest message and the `thread_id` are required; Lakebase supplies everything else.
- Because Lakebase is a managed Postgres database, checkpoints survive endpoint restarts and are queryable directly with SQL if you need to audit or debug conversations.

## Next steps: register and deploy
To make this short-term memory work in production, log the agent with `mlflow.pyfunc.log_model`, register it to Unity Catalog, and deploy it with `databricks-agents`. Once deployed, callers pass the `thread_id` through `extra_body={"custom_inputs": {"thread_id": thread_id}}` (or simply rely on `ChatContext.conversation_id` from the Playground / Review App). See [Custom Agents on Model Serving](https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent-model-serving) and [Query an agent deployed on Databricks](https://docs.databricks.com/aws/en/generative-ai/agent-framework/query-agent) for the full deployment and querying workflow.

For more advanced short-term memory patterns — such as resuming or branching a conversation from any prior checkpoint ("time travel") — see [AI agent memory (Model Serving)](https://docs.databricks.com/aws/en/generative-ai/agent-framework/stateful-agents-model-serving#-short-term-memory-time-travel).